# section_category가 result인 데이터만 다른이름의 csv로 저장하기

In [1]:
import pandas as pd

df = pd.read_csv("raw_dataset_reference.csv")
df.head(10)

,section_id,pmid,topic_category,section_title,section_text,path,section_category,article_category,fig_ids,table_ids
0,38598310_sec1,38598310,Protein Structure & Enzyme Engineering,Introduction,"The “sunshine vitamin”, vitamin D3(VitD3), is ...",Introduction,introduction,research,161183008895;384174878961;470307513129,NaN
1,38598310_sec4,38598310,Protein Structure & Enzyme Engineering,Conformational Search,All stationary-point geometries(equilibrium co...,Calculation Method > Conformational Search,method,research,678947196634,NaN
2,38598310_sec8,38598310,Protein Structure & Enzyme Engineering,Accessible Conformers,Based on the findings from Furche and co-worke...,Results and Discussion > Accessible Conformers,result,research,683438586433,NaN
3,38598310_sec9,38598310,Protein Structure & Enzyme Engineering,Conformational Analysis,The conformational exploration using TorsiFlex...,Results and Discussion > Conformational Analysis,result,research,426700642544;798125557350,NaN
4,38598310_sec10,38598310,Protein Structure & Enzyme Engineering,Thermal Rate Constants and Catalytic Effect,Our MD simulations show that all conformers mu...,Results and Discussion > Thermal Rate Constant...,result,research,210276443294,tbl1
5,36779817_sec4,36779817,Protein Structure & Enzyme Engineering,FUNCTIONAL ROLE OF FOLDING INTERMEDIATE IN PRO...,4 Serpins(serine protease inhibitors) provide ...,FUNCTIONAL ROLE OF FOLDING INTERMEDIATE IN PRO...,main,review,953571766348,NaN
6,36779817_sec6,36779817,Protein Structure & Enzyme Engineering,PHARMACOLOGICAL VALUE OF TARGETING FOLDING INT...,6 Conventional rational drug discovery is prim...,PHARMACOLOGICAL VALUE OF TARGETING FOLDING INT...,main,review,282551023184,NaN
7,40919504_sec2,40919504,Protein Structure & Enzyme Engineering,Measuring stability,2 Many single-domain globular proteins undergo...,Measuring stability,main,review,128964201360,NaN
8,39739814_sec2,39739814,Protein Structure & Enzyme Engineering,Identification of MALT1 as a Modulator of GPX4...,To evaluate the potential of leveraging ferrop...,Results > Identification of MALT1 as a Modulat...,result,research,276749254860,NaN
9,39739814_sec3,39739814,Protein Structure & Enzyme Engineering,MALT1 Stabilizes GPX4 through Cleavage of E3 U...,To investigate the mechanisms by which MALT1 r...,Results > MALT1 Stabilizes GPX4 through Cleava...,result,research,145411149905,NaN


In [2]:
result_df = df[df["section_category"] == "result"].copy()
result_df.head(10)

,section_id,pmid,topic_category,section_title,section_text,path,section_category,article_category,fig_ids,table_ids
2,38598310_sec8,38598310,Protein Structure & Enzyme Engineering,Accessible Conformers,Based on the findings from Furche and co-worke...,Results and Discussion > Accessible Conformers,result,research,683438586433,NaN
3,38598310_sec9,38598310,Protein Structure & Enzyme Engineering,Conformational Analysis,The conformational exploration using TorsiFlex...,Results and Discussion > Conformational Analysis,result,research,426700642544;798125557350,NaN
4,38598310_sec10,38598310,Protein Structure & Enzyme Engineering,Thermal Rate Constants and Catalytic Effect,Our MD simulations show that all conformers mu...,Results and Discussion > Thermal Rate Constant...,result,research,210276443294,tbl1
8,39739814_sec2,39739814,Protein Structure & Enzyme Engineering,Identification of MALT1 as a Modulator of GPX4...,To evaluate the potential of leveraging ferrop...,Results > Identification of MALT1 as a Modulat...,result,research,276749254860,NaN
9,39739814_sec3,39739814,Protein Structure & Enzyme Engineering,MALT1 Stabilizes GPX4 through Cleavage of E3 U...,To investigate the mechanisms by which MALT1 r...,Results > MALT1 Stabilizes GPX4 through Cleava...,result,research,145411149905,NaN
10,39739814_sec4,39739814,Protein Structure & Enzyme Engineering,MALT1 Inhibition Triggers Ferroptosis in Liver...,The crucial regulatory role of the MALT1–RC3H1...,Results > MALT1 Inhibition Triggers Ferroptosi...,result,research,614429346520,NaN
11,39739814_sec5,39739814,Protein Structure & Enzyme Engineering,MALT1 Inhibitor Down-Regulates GPX4 through Ub...,Quantitative proteomics analysis revealed that...,Results > MALT1 Inhibitor Down-Regulates GPX4 ...,result,research,245734622300,NaN
12,39739814_sec6,39739814,Protein Structure & Enzyme Engineering,MALT1 Inhibition Is Synergistic with Sorafenib...,Given that many liver cancer cell lines demons...,Results > MALT1 Inhibition Is Synergistic with...,result,research,659650355594,NaN
13,39742998_sec2,39742998,Protein Structure & Enzyme Engineering,H80 and E166 are essential for RNase J2 activity,The active site of RNase J2 has a single metal...,Results > H80 and E166 are essential for RNase...,result,research,497300792137,NaN
14,39742998_sec4,39742998,Protein Structure & Enzyme Engineering,Influence of the active site dynamics on RNase...,The RT-FeDEx assay revealed an unanticipated f...,Results > Influence of the active site dynamic...,result,research,356948113015,NaN


In [3]:
result_df.to_csv("section_category_result.csv", index=False)

# final_sft_dataset_*.jsonl 을 train.jsonl, test.jsonl 파일로 나누기

In [ ]:
import glob
import json
import random
import os

# 절대 경로 기준으로 설정
base_dir = "sllm/datasets/generated"

# final_sft_dataset_*.jsonl 파일 찾기
jsonl_files = glob.glob(f"{base_dir}/final_sft_dataset_*.jsonl")
all_data = []
for fname in jsonl_files:
    with open(fname, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                all_data.append(json.loads(line))

# 셔플
random.shuffle(all_data)

# 7:3으로 split
n_total = len(all_data)
split_idx = int(n_total * 0.7)
train_data = all_data[:split_idx]
test_data = all_data[split_idx:]

# 저장 경로 명확하게 설정
train_save_path = f"sllm/datasets/train/train.jsonl"
test_save_path = f"sllm/datasets/eval/test.jsonl"

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(train_save_path), exist_ok=True)

with open(train_save_path, "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(test_save_path, "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Total: {n_total}, Train: {len(train_data)}, Test: {len(test_data)}")
print(f"Train saved to: {train_save_path}")
print(f"Test saved to: {test_save_path}")


Total: 10, Train: 7, Test: 3
Train saved to: ./train/train.jsonl
Test saved to: ./eval/test.jsonl
